In [11]:
# %pip install ipywidgets --upgrade --quiet

import sempy.fabric as fabric
import pandas as pd
import json, re, os, hashlib
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
import warnings
warnings.filterwarnings("ignore")

try:
    import sempy
    _sempy_ver = sempy.__version__
except Exception:
    _sempy_ver = "loaded"

spark = SparkSession.builder.getOrCreate()

print(f"✅  SemPy      : {_sempy_ver}")
print(f"✅  ipywidgets : {widgets.__version__}")
print(f"✅  Pandas     : {pd.__version__}")
print(f"✅  Spark      : {spark.version}")


# ─────────────────────────────────────────────────────────────────────────────
# CONSTANTS
# ─────────────────────────────────────────────────────────────────────────────
LAKEHOUSE_ROOT  = "/lakehouse/default"
TABLES_PATH     = f"{LAKEHOUSE_ROOT}/Tables"
SNAPSHOT_PREFIX = "sm_snapshot"

# Delta table names (written to Lakehouse Tables — queryable in Power BI)
TBL_MEASURES      = f"{SNAPSHOT_PREFIX}_measures"
TBL_COLUMNS       = f"{SNAPSHOT_PREFIX}_columns"
TBL_RELATIONSHIPS = f"{SNAPSHOT_PREFIX}_relationships"
TBL_TABLES        = f"{SNAPSHOT_PREFIX}_tables"
TBL_ROLES         = f"{SNAPSHOT_PREFIX}_roles"
TBL_RUNS          = f"{SNAPSHOT_PREFIX}_runs"       # run registry

# Shared state
DATASET_NAME   =  'AdventureWorks Sales'
WORKSPACE_NAME = 'dev'


StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 13, Finished, Available, Finished, False)

✅  SemPy      : 0.11.0
✅  ipywidgets : 8.1.2
✅  Pandas     : 2.1.4
✅  Spark      : 3.5.5.5.4.20260211.1


In [12]:
run_id=datetime.now().strftime("%Y%m%d_%H%M%S")
# _safe_name=re.sub(r"[^a-zA-Z0-9_]", "_", s or "unknown")

StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 14, Finished, Available, Finished, False)

In [13]:
from IPython.display import display, HTML
import pandas as pd
import uuid

def display_fabric_table(df: pd.DataFrame, title: str = "", col_min_width: int = 120, max_cell_chars: int = 30):
    """
    Renders a DataFrame as a Fabric-branded HTML table.
    Fixes column text mixing via min-width, ellipsis truncation, and isolated scroll.

    Parameters
    ----------
    df             : pandas DataFrame to render
    title          : optional title string shown above the table
    col_min_width  : minimum column width in px (default 120)
    max_cell_chars : truncate cell text beyond this length with ellipsis + tooltip
    """

    # ── Fabric Green palette ─────────────────────────────────────────────────
    FABRIC_GREEN_DARK   = "#0E7A6A"
    FABRIC_GREEN_MID    = "#1AAF87"
    FABRIC_GREEN_LIGHT  = "#A8EDD8"
    FABRIC_GREEN_ACCENT = "#3ECFAA"
    ROW_ALT             = "#F0FAF7"
    ROW_HOVER           = "#D6F5EC"
    BORDER_COLOR        = "#C8EAE1"
    TEXT_DARK           = "#201F1E"
    TEXT_HEADER         = "#FFFFFF"
    FONT                = "Segoe UI, Segoe UI Web, -apple-system, sans-serif"

    HEADER_GRADIENT = (
        f"linear-gradient(135deg, "
        f"{FABRIC_GREEN_DARK} 0%, "
        f"{FABRIC_GREEN_MID} 40%, "
        f"{FABRIC_GREEN_ACCENT} 70%, "
        f"{FABRIC_GREEN_LIGHT} 100%)"
    )
    TITLE_GRADIENT = (
        f"linear-gradient(135deg, #0A5C52 0%, {FABRIC_GREEN_DARK} 50%, {FABRIC_GREEN_MID} 100%)"
    )

    # ── Auto min-width: use the longest header text to set a sensible floor ──
    def col_width(col_name: str) -> int:
        char_width  = max(len(str(col_name)), 8)   # at least 8 chars wide
        pixel_width = char_width * 9               # ~9px per char in Segoe UI 12px
        return max(pixel_width, col_min_width)

    # ── Build <colgroup> for per-column widths ───────────────────────────────
    colgroup = "<colgroup>" + "".join(
        f'<col style="min-width:{col_width(c)}px; width:{col_width(c)}px;">'
        for c in df.columns
    ) + "</colgroup>"

    # ── Header row ───────────────────────────────────────────────────────────
    header_cells = "".join(
        f'<th title="{col}">{col}</th>' for col in df.columns
    )

    # ── Body rows — truncate long values and add tooltip ────────────────────
    def fmt_cell(val) -> str:
        text = "" if pd.isna(val) else str(val)
        if len(text) > max_cell_chars:
            truncated = text[:max_cell_chars] + "…"
            return f'<td title="{text}">{truncated}</td>'
        return f'<td title="{text}">{text}</td>'

    body_rows = ""
    for i, (_, row) in enumerate(df.iterrows()):
        bg    = "white" if i % 2 == 0 else ROW_ALT
        cells = "".join(fmt_cell(v) for v in row)
        body_rows += f'<tr style="background:{bg}">{cells}</tr>\n'

    # ── Title block ──────────────────────────────────────────────────────────
    title_html = ""
    if title:
        title_html = f"""
        <div style="
            background: {TITLE_GRADIENT};
            color: {FABRIC_GREEN_LIGHT};
            font-family: {FONT};
            font-size: 15px;
            font-weight: 700;
            letter-spacing: 0.4px;
            padding: 11px 18px;
            border-radius: 8px 8px 0 0;
            border-left: 4px solid {FABRIC_GREEN_LIGHT};
        ">{title}</div>"""

    # ── Row / col count badge ────────────────────────────────────────────────
    badge = f"""
    <div style="
        font-family: {FONT};
        font-size: 11px;
        color: {FABRIC_GREEN_MID};
        font-weight: 600;
        padding: 6px 16px 8px;
        background: white;
        border-radius: 0 0 8px 8px;
        border-top: 2px solid {BORDER_COLOR};
    ">
        {len(df):,} row{"s" if len(df) != 1 else ""}&nbsp;·&nbsp;
        {len(df.columns)} column{"s" if len(df.columns) != 1 else ""}
    </div>"""

    # ── Scoped CSS (uuid prevents bleed across tables) ───────────────────────
    tbl_id = f"fabric_{uuid.uuid4().hex[:8]}"

    html = f"""
    <style>
        #{tbl_id} {{
            font-family: {FONT};
            border: 1px solid {BORDER_COLOR};
            border-radius: 8px;
            overflow: hidden;
            box-shadow: 0 3px 12px rgba(14,122,106,0.15);
            margin: 14px 0;
        }}
        /* ── Scroll wrapper: THIS is what isolates overflow ── */
        #{tbl_id} .tbl-scroll {{
            overflow-x: auto;
            overflow-y: auto;
            max-height: 520px;      /* vertical scroll for long DataFrames  */
            -webkit-overflow-scrolling: touch;
        }}
        #{tbl_id} table {{
            width: max-content;     /* let table be as wide as it needs      */
            min-width: 100%;
            border-collapse: collapse;
            font-size: 13px;
            color: {TEXT_DARK};
            table-layout: fixed;    /* honour <col> widths strictly          */
        }}
        /* ── Sticky header so columns stay labelled when scrolling ── */
        #{tbl_id} thead tr {{
            background: {HEADER_GRADIENT};
            position: sticky;
            top: 0;
            z-index: 2;
        }}
        #{tbl_id} th {{
            color: {TEXT_HEADER};
            font-weight: 700;
            font-size: 11px;
            text-transform: uppercase;
            letter-spacing: 0.6px;
            padding: 10px 14px;
            text-align: left;
            border-right: 1px solid rgba(255,255,255,0.18);
            /* KEY FIX: never wrap, clip with ellipsis */
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
            text-shadow: 0 1px 2px rgba(0,0,0,0.2);
        }}
        #{tbl_id} th:last-child {{ border-right: none; }}

        #{tbl_id} td {{
            padding: 8px 14px;
            border-bottom: 1px solid {BORDER_COLOR};
            border-right: 1px solid {BORDER_COLOR};
            /* KEY FIX: clip text within its own column box */
            white-space: nowrap;
            overflow: hidden;
            text-overflow: ellipsis;
            max-width: 0;           /* required for ellipsis in table-layout:fixed */
        }}
        #{tbl_id} td:last-child {{ border-right: none; }}
        #{tbl_id} tr:last-child td {{ border-bottom: none; }}

        #{tbl_id} tbody tr:hover td {{
            background: {ROW_HOVER} !important;
            transition: background 0.15s ease;
        }}
    </style>

    <div id="{tbl_id}">
        {title_html}
        <div class="tbl-scroll">
            <table>
                {colgroup}
                <thead><tr>{header_cells}</tr></thead>
                <tbody>{body_rows}</tbody>
            </table>
        </div>
        {badge}
    </div>
    """

    display(HTML(html))

StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 15, Finished, Available, Finished, False)

In [14]:
def take_snapshot_measures(run_id: str):
        
    try:
            df = fabric.list_measures(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
            # print(df.shape)
    except Exception:
            df = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME, dax_string="""
                SELECT [TableID],[Name],[Expression],[Description],
                    [FormatString],[IsHidden]
                FROM $SYSTEM.TMSCHEMA_MEASURES
            """)
    if df.empty:
            df_measures=pd.DataFrame()
        # Normalise columns
    else:
        col_map = {
                "Measure Name":"Name","Table Name":"Table","Measure Expression":"Expression",
                "Description":"Description","Format String":"FormatString","Hidden":"IsHidden",
            }
        df = df.rename(columns={k:v for k,v in col_map.items() if k in df.columns})
        keep = [c for c in ["Name","Table","Expression","Description","FormatString","IsHidden"]
                    if c in df.columns]
        # display_fabric_table(df)
        return df[keep]


StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 16, Finished, Available, Finished, False)

In [15]:
def take_snapshot_columns(run_id: str):
    try:
            df = fabric.list_columns(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
            # print(df.shape)
    except Exception:
            df = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME, dax_string="SELECT [Name],[Type],[DataType],[Expression],[IsHidden] FROM $SYSTEM.TMSCHEMA_COLUMNS"
        )
    if df.empty:
            # df_measures=pd.DataFrame()
            print('There are no Columns in this Semantic Model')
        # Normalise columns
    else:
        col_map = {
            "Column Name":"Name","Table Name":"Table","Data Type":"DataType",
            "Column Type":"ColumnType","Expression":"Expression","Hidden":"IsHidden",
        }
        df = df.rename(columns={k:v for k,v in col_map.items() if k in df.columns})
        keep = [c for c in ["Name","Table","DataType","ColumnType","Expression","IsHidden"]
                if c in df.columns]
        # print(df[keep])
        # display_fabric_table(df,"Columns")
        return df[keep]

StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 17, Finished, Available, Finished, False)

In [16]:
def take_snapshot_relationships(run_id: str):
    try:
            df = fabric.list_relationships(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
            # print(df.shape)
    except Exception:
            df = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME, dax_string="""SELECT [FromTableID],[ToTableID],[FromCardinality],
                                [ToCardinality],[CrossFilteringBehavior],[IsActive]
                        FROM $SYSTEM.TMSCHEMA_RELATIONSHIPS""")
    if df.empty:
            # df_measures=pd.DataFrame()
            print('There are no relationships in this Semantic Model')
        # Normalise columns
    else:
        col_map = {
            "From Table":"FromTable","From Column":"FromColumn",
            "To Table":"ToTable","To Column":"ToColumn",
            "From Cardinality":"FromCardinality","To Cardinality":"ToCardinality",
            "Cross Filtering Behavior":"CrossFilter","Active":"IsActive",
        }
        df = df.rename(columns={k:v for k,v in col_map.items() if k in df.columns})
        keep = [c for c in ["FromTable","FromColumn","ToTable","ToColumn",
                            "FromCardinality","ToCardinality","CrossFilter","IsActive"]
                if c in df.columns]
        # print(df[keep])
        # display_fabric_table(df,"Relationships")
        
        return df[keep]

StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 18, Finished, Available, Finished, False)

In [31]:
def take_snapshot_tables(run_id: str):
    """Capture all tables."""
    try:
        df = fabric.list_tables(dataset=DATASET_NAME, workspace=WORKSPACE_NAME)
        # print(df.shape)
    except Exception:
        df = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME, 
                dax_string="SELECT [Name],[IsHidden],[StorageMode] FROM $SYSTEM.TMSCHEMA_TABLES")
    if df.empty:
        print('There are no tables in this Semantic Model')
    else:
        col_map = {"Table Name":"Name","Hidden":"IsHidden","Storage Mode":"StorageMode"}
        df = df.rename(columns={k:v for k,v in col_map.items() if k in df.columns})
        keep = [c for c in ["Name","IsHidden","StorageMode"] if c in df.columns]
        # print(df[keep])
        # display_fabric_table(df,"Tables")
        return df[keep]

StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 33, Finished, Available, Finished, False)

In [18]:
def take_snapshot_roles(run_id: str):
    """Capture RLS roles and filter expressions."""
    roles = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME,
                dax_string="SELECT [Name],[ModelPermission],[ID] FROM $SYSTEM.TMSCHEMA_ROLES")
    # print('roles done')             
    filters = fabric.evaluate_dax(
                dataset=DATASET_NAME, workspace=WORKSPACE_NAME, 
                dax_string="""SELECT [RoleID],[MemberName],[MemberType],[IdentityProvider]
                        FROM $SYSTEM.TMSCHEMA_ROLE_MEMBERSHIPS""")
    if roles.empty:
        print('There are no roles in this Semantic Model')
    if not filters.empty and "RoleID" in filters.columns:
        # Join on RoleID if possible, else just return roles
        try:
            merged =roles.merge(filters, right_on="RoleID",left_on='ID', how="left")
            keep = [c for c in ["RoleID","Name","IdentityProvider","MemberName"] if c in merged.columns]
            # print(merged[keep])
            # display_fabric_table(merged[keep],"Roles")
            return merged[keep]
        except Exception:
            pass
    else:
        print('either filter is empty ot roleid is not found')


StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 20, Finished, Available, Finished, False)

In [28]:
# ─────────────────────────────────────────────────────────────────────────────
# SNAPSHOT FUNCTIONS — capture metadata from Semantic Link
# ─────────────────────────────────────────────────────────────────────────────

def _run_id() -> str:
    return datetime.now().strftime("%Y%m%d_%H%M%S")

def _safe_name(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_]", "_", s or "unknown")

def _dmv(query: str) -> pd.DataFrame:
    try:
        return fabric.evaluate_dax(
            dataset=DATASET_NAME, workspace=WORKSPACE_NAME, dax_string=query)
    except Exception:
        return pd.DataFrame()

def _write_delta(df: pd.DataFrame, table_name: str, run_id: str,
                 model: str, workspace: str):
    """Append a pandas DataFrame to a Lakehouse Delta table with run metadata."""
    if df.empty:
        return
    df = df.copy()
    df["_run_id"]    = run_id
    df["_model"]     = model
    df["_workspace"] = workspace
    df["_timestamp"] = datetime.now()
    sdf = spark.createDataFrame(df.astype(str))
    sdf.write.format("delta").mode("append") \
       .option("mergeSchema","true") \
       .saveAsTable(table_name)
def take_snapshot() -> str:
    """Run all snapshot functions and write to Delta tables. Returns run_id."""
    run_id = _run_id()
    model  = DATASET_NAME
    ws     = WORKSPACE_NAME

    steps = [
        ("Measures",      take_snapshot_measures,      TBL_MEASURES),
        ("Columns",        take_snapshot_columns,       TBL_COLUMNS),
        ("Relationships",  take_snapshot_relationships, TBL_RELATIONSHIPS),
        ("Tables",         take_snapshot_tables,        TBL_TABLES),
        ("RLS Roles",      take_snapshot_roles,         TBL_ROLES),
    ]

    results = {}
    for label, fn, tbl in steps:
        try:
            df = fn(run_id)
            _write_delta(df, tbl, run_id, model, ws)
            results[label] = len(df)
        except Exception as e:
            results[label] = f"⚠️ {e}"

    # Write run registry entry
    run_row = pd.DataFrame([{
        "run_id"   : run_id,
        "model"    : model,
        "workspace": ws,
        "timestamp": datetime.now(),
        "measures" : str(results.get("Measures","?")),
        "columns"  : str(results.get("Columns","?")),
        "relationships": str(results.get("Relationships","?")),
        "tables"   : str(results.get("Tables","?")),
        "roles"    : str(results.get("RLS Roles","?")),
    }])
    sdf = spark.createDataFrame(run_row.astype(str))
    sdf.write.format("delta").mode("append") \
       .option("mergeSchema","true").saveAsTable(TBL_RUNS)

    return run_id, results
# ─────────────────────────────────────────────────────────────────────────────
# DIFF FUNCTIONS — compare two snapshots
# ─────────────────────────────────────────────────────────────────────────────
def _get_last_two_runs() -> tuple[str, str] | tuple[None, None]:
    """Return (latest_run_id, previous_run_id) for the current model."""
    try:
        df = spark.sql(f"""
            SELECT run_id, timestamp FROM {TBL_RUNS}
            WHERE model = '{DATASET_NAME}' AND workspace = '{WORKSPACE_NAME}'
            ORDER BY timestamp DESC LIMIT 2
        """).toPandas()
        if len(df) < 2:
            return df["run_id"].iloc[0] if len(df) == 1 else None, None
        return df["run_id"].iloc[0], df["run_id"].iloc[1]
    except Exception:
        return None, None

def _load_snapshot(table: str, run_id: str) -> pd.DataFrame:
    try:
        df = spark.sql(f"""
            SELECT * FROM {table}
            WHERE _run_id = '{run_id}'
            AND _model = '{DATASET_NAME}'
            AND _workspace = '{WORKSPACE_NAME}'
        """).toPandas()
        # Drop internal cols
        return df.drop(columns=[c for c in ["_run_id","_model","_workspace","_timestamp"]
                                  if c in df.columns], errors="ignore")
    except Exception:
        return pd.DataFrame()

def _hash_row(row: pd.Series) -> str:
    return hashlib.md5("|".join(str(v) for v in row.values).encode()).hexdigest()

def _diff_dataframes(old_df: pd.DataFrame, new_df: pd.DataFrame,
                     key_cols: list[str], compare_cols: list[str]) -> pd.DataFrame:
    """
    Compare two snapshots. Returns a DataFrame with a 'Change' column:
    ADDED / REMOVED / MODIFIED (only rows with ADDED/REMOVED/MODIFIED returned).
    """
    if old_df.empty and new_df.empty:
        return pd.DataFrame()

    # Ensure key columns exist in both
    key_cols = [k for k in key_cols if k in (old_df.columns if not old_df.empty else [])
                                          and k in (new_df.columns if not new_df.empty else [])]
    if not key_cols:
        return pd.DataFrame()

    old_df = old_df.copy().fillna("").astype(str)
    new_df = new_df.copy().fillna("").astype(str)

    old_keys = set(map(tuple, old_df[key_cols].values.tolist()))
    new_keys = set(map(tuple, new_df[key_cols].values.tolist()))

    added   = new_df[new_df[key_cols].apply(tuple, axis=1).isin(new_keys - old_keys)].copy()
    removed = old_df[old_df[key_cols].apply(tuple, axis=1).isin(old_keys - new_keys)].copy()

    added["Change"]   = "ADDED"
    removed["Change"] = "REMOVED"

    # Modified: same key, different values in compare_cols
    common_keys = old_keys & new_keys
    modified_rows = []
    for key in common_keys:
        key_filter = lambda df: df[df[key_cols].apply(tuple, axis=1) == key]
        old_row = key_filter(old_df)
        new_row = key_filter(new_df)
        if old_row.empty or new_row.empty:
            continue
        for col in compare_cols:
            if col not in old_row.columns or col not in new_row.columns:
                continue
            ov = old_row[col].iloc[0]
            nv = new_row[col].iloc[0]
            if ov != nv:
                base = {k: new_row[k].iloc[0] for k in key_cols}
                base["Field"]          = col
                base["Old Value"]      = ov[:200] + ("…" if len(ov) > 200 else "")
                base["New Value"]      = nv[:200] + ("…" if len(nv) > 200 else "")
                base["Change"]         = "MODIFIED"
                modified_rows.append(base)

    modified_df = pd.DataFrame(modified_rows) if modified_rows else pd.DataFrame()

    parts = [p for p in [added, removed, modified_df] if not p.empty]
    if not parts:
        return pd.DataFrame()
    return pd.concat(parts, ignore_index=True)

def compute_diff(run_new: str, run_old: str) -> dict:
    """Compute diffs across all artifact types."""
    diffs = {}

    # Measures
    old_m = _load_snapshot(TBL_MEASURES, run_old)
    new_m = _load_snapshot(TBL_MEASURES, run_new)
    diffs["Measures"] = _diff_dataframes(
        old_m, new_m,
        key_cols     = ["Name","Table"],
        compare_cols = ["Expression","Description","FormatString","IsHidden"],
    )
    if diffs["Measures"].empty:
        print('there is no difference between the 2 versions for Measures')
    # Columns
    old_c = _load_snapshot(TBL_COLUMNS, run_old)
    new_c = _load_snapshot(TBL_COLUMNS, run_new)
    diffs["Columns"] = _diff_dataframes(
        old_c, new_c,
        key_cols     = ["Name","Table"],
        compare_cols = ["DataType","ColumnType","Expression","IsHidden"],
    )
    if diffs["Columns"].empty:
        print('there is no difference between the 2 versions for Columns')

    # Relationships
    old_r = _load_snapshot(TBL_RELATIONSHIPS, run_old)
    new_r = _load_snapshot(TBL_RELATIONSHIPS, run_new)
    diffs["Relationships"] = _diff_dataframes(
        old_r, new_r,
        key_cols     = ["FromTable","FromColumn","ToTable","ToColumn"],
        compare_cols = ["FromCardinality","ToCardinality","CrossFilter","IsActive"],
    )
    if diffs["Relationships"].empty:
        print('there is no difference between the 2 versions for Relationships')
    # Tables
    old_t = _load_snapshot(TBL_TABLES, run_old)
    new_t = _load_snapshot(TBL_TABLES, run_new)
    diffs["Tables"] = _diff_dataframes(
        old_t, new_t,
        key_cols     = ["Name"],
        compare_cols = ["IsHidden","StorageMode"],
    )
    if diffs["Tables"].empty:
        print('there is no difference between the 2 versions for Tables')

    # Roles
    old_rl = _load_snapshot(TBL_ROLES, run_old)
    new_rl = _load_snapshot(TBL_ROLES, run_new)
    diffs["RLS Roles"] = _diff_dataframes(
        old_rl, new_rl,
        key_cols     = ["Name"],
        compare_cols = ["ModelPermission","FilterExpression"],
    )
    if diffs["RLS Roles"].empty:
        print('there is no difference between the 2 versions for Roles')
    return diffs


StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 30, Finished, Available, Finished, False)

In [32]:
# DATASET_NAME='SampleSemanticModel'
# WORKSPACE_NAME='dev'
# snapshot_relationships()
# snapshot_columns('id')
print(take_snapshot())
# fabric.list_workspaces()



StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 34, Finished, Available, Finished, False)

('20260419_190811', {'Measures': 9, 'Columns': 57, 'Relationships': 12, 'Tables': 8, 'RLS Roles': 6})


In [29]:
run_new,run_old=_get_last_two_runs()
compute_diff(run_new,run_old)

StatementMeta(, 0407474b-81e0-45f4-8768-81783ad42b41, 31, Finished, Available, Finished, False)

there is no difference between the 2 versions for Measures
there is no difference between the 2 versions for Columns
there is no difference between the 2 versions for Relationships
there is no difference between the 2 versions for tables
there is no difference between the 2 versions for Roles


{'Measures': Empty DataFrame
 Columns: []
 Index: [],
 'Columns': Empty DataFrame
 Columns: []
 Index: [],
 'Relationships': Empty DataFrame
 Columns: []
 Index: [],
 'Tables': Empty DataFrame
 Columns: []
 Index: [],
 'RLS Roles': Empty DataFrame
 Columns: []
 Index: []}